## Importación de módulos

In [68]:
import random
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import DistilBertForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from evaluate import load
import os

## Implementación de Clase basada en polinomios de Chebishev

In [69]:
class ChebyshevPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super(ChebyshevPositionalEncoding, self).__init__()
        self.d_model = d_model
        self.max_len = max_len
        
        pe = torch.zeros(max_len, d_model)

        if max_len > 1:
            p = torch.arange(0, max_len, dtype=torch.float32)
            x = -1.0 + 2.0 * p / (max_len - 1)
        else:
            x = torch.zeros(1, dtype=torch.float32)
            
        arccos_x = torch.acos(x) 
        k = torch.arange(0, d_model, dtype=torch.float32)
        
        angles = arccos_x.unsqueeze(1) * k.unsqueeze(0)
        pe = torch.cos(angles)
        
        pe = pe / math.sqrt(d_model)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, input_ids_or_position_ids, *args, **kwargs):
        seq_len = input_ids_or_position_ids.size(1)
        
        return self.pe[:, :seq_len, :]

## Functions

### Function to set the seeder on each training test for the baseline and the proposed model

Changing the seed for each tuning run ensures randomness, forcing each training session to use unique values and thereby preventing anomalous results.

In [70]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

### Function to initialize the instance of the model, receiving a flag to switch between the baseline and the proposed model

In [71]:
def build_model(use_chebyshev: bool, device: torch.device):
    model_name = "distilbert-base-uncased"
    model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    if use_chebyshev:
        d_model = model.config.hidden_size
        chebyshev_encoder = ChebyshevPositionalEncoding(d_model=d_model, max_len=512)
        model.distilbert.embeddings.position_embeddings = chebyshev_encoder
        
    return model.to(device)

Function to train and evaluate each model several times

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, device, epochs=2):
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    acc_metric = load("accuracy")
    f1_metric = load("f1")
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
    # Evaluación en Test
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            acc_metric.add_batch(predictions=preds, references=batch["labels"])
            f1_metric.add_batch(predictions=preds, references=batch["labels"])
            
    return acc_metric.compute()["accuracy"], f1_metric.compute()["f1"]

## Unique loading data

The base model, the tokenizer, the dataset, and the samples are initizalized

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

raw_dataset = load_dataset("stanfordnlp/imdb")
train_subset = raw_dataset["train"].shuffle(seed=42).select(range(5000))
test_subset = raw_dataset["test"].shuffle(seed=42).select(range(1000))

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_subset.map(tokenize_fn, batched=True).remove_columns(["text"]).rename_column("label", "labels")
tokenized_test = test_subset.map(tokenize_fn, batched=True).remove_columns(["text"]).rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(tokenized_train, batch_size=16, shuffle=True, collate_fn=data_collator)
test_loader = DataLoader(tokenized_test, batch_size=16, collate_fn=data_collator)

## Testing Execution

### Baseline Model Experiment

In [75]:
def run_baseline_experiment(seed, train_loader, test_loader, device, save_base_dir="./modelos_tesis"):
    set_seed(seed)
    model = build_model(use_chebyshev=False, device=device)
    acc, f1 = train_and_evaluate(model, train_loader, test_loader, device)
    
    save_path = os.path.join(save_base_dir, f"baseline_seed_{seed}")
    model.save_pretrained(save_path)
    
    return {
        "model_type": "Baseline",
        "seed": seed,
        "accuracy": acc,
        "f1_score": f1,
        "model_path": save_path
    }

### Chebyshev Model Experiment

In [76]:
def run_chebyshev_experiment(seed, train_loader, test_loader, device, save_base_dir="./modelos_tesis"):
    set_seed(seed)
    model = build_model(use_chebyshev=True, device=device)
    acc, f1 = train_and_evaluate(model, train_loader, test_loader, device)
    
    save_path = os.path.join(save_base_dir, f"chebyshev_seed_{seed}")
    os.makedirs(save_path, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_path, "model.pt"))
    
    return {
        "model_type": "Chebyshev",
        "seed": seed,
        "accuracy": acc,
        "f1_score": f1,
        "model_path": save_path
    }

## Execution

In [77]:
seeds = [42, 123, 999]

results = []

print("=== INICIANDO EXPERIMENTO COMPARATIVO MULTI-SEMILLA ===")

for seed in seeds:
    print(f"\n---> Corriendo experimentos para semilla {seed}...")
    
    res_a = run_baseline_experiment(seed, train_loader, test_loader, device)
    results.append(res_a)

    res_b = run_chebyshev_experiment(seed, train_loader, test_loader, device)
    results.append(res_b)


df_results = pd.DataFrame(results)
df_results.to_csv("./modelos_tesis/resultados_consolidados.csv", index=False)

print("\n--- MATRIZ CONSOLIDADA DE RESULTADOS ---")
print(df_results)

summary = df_results.groupby("model_type")[["accuracy", "f1_score"]].agg(["mean", "std"])
print(summary)

=== INICIANDO EXPERIMENTO COMPARATIVO MULTI-SEMILLA ===

---> Corriendo experimentos para semilla 42...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- MATRIZ CONSOLIDADA DE RESULTADOS ---
  model_type  seed  accuracy  f1_score                          model_path
0   Baseline    42     0.852  0.856031    ./modelos_tesis/baseline_seed_42
1  Chebyshev    42     0.810  0.788419   ./modelos_tesis/chebyshev_seed_42
2   Baseline   123     0.839  0.838191   ./modelos_tesis/baseline_seed_123
3  Chebyshev   123     0.828  0.824490  ./modelos_tesis/chebyshev_seed_123
4   Baseline   999     0.841  0.842105   ./modelos_tesis/baseline_seed_999
5  Chebyshev   999     0.808  0.809524  ./modelos_tesis/chebyshev_seed_999
            accuracy            f1_score          
                mean       std      mean       std
model_type                                        
Baseline    0.844000  0.007000  0.845442  0.009377
Chebyshev   0.815333  0.011015  0.807477  0.018122
